<a href="https://colab.research.google.com/github/likhitha-1231/week2_Final-/blob/main/Project3_Smart_Analytics_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_daily_reports/01-01-2021.csv"
df = pd.read_csv(url)
df_clean = df[['Country_Region', 'Confirmed', 'Deaths', 'Recovered', 'Active']].groupby('Country_Region').sum().reset_index()
df_clean.to_csv("covid_data.csv", index=False)

In [ ]:
!pip install -q gradio pandas plotly numpy
import pandas as pd
import plotly.express as px
import gradio as gr
import numpy as np

In [ ]:
def analyze_csv(file_obj):
    if file_obj is None:
        return "No file uploaded", None, None, None, None, None
    df = pd.read_csv(file_obj.name)
    preview = df.head(5).to_html()
    missing = pd.DataFrame(df.isnull().sum(), columns=["Missing Count"]).to_html()
    stats = df.describe().reset_index().to_html()
    df_filtered = df[df['Confirmed'] > 0].copy()
    fig1 = px.scatter(df_filtered, x='Confirmed', y='Deaths', hover_name='Country_Region',
                      color='Country_Region', title="1. Global Correlation: Confirmed Cases vs Deaths (Country-wise Colors)")
    fig1.update_layout(showlegend=False)
    fig2 = px.box(df_filtered, x='Confirmed', log_x=True, points="all", hover_name='Country_Region',
                  title="2. Global Data Spread: Distribution of Confirmed Cases (Log Scale)")
    fig2.update_traces(marker=dict(color='#ff6b6b'))
    top_10 = df_filtered.nlargest(10, 'Confirmed')
    fig3 = px.bar(top_10, x='Country_Region', y='Confirmed', color='Deaths',
                  title="3. Bar Chart: Top 10 Worst Affected Countries Globally")

    return preview, missing, stats, fig1, fig2, fig3

In [ ]:
with gr.Blocks() as tool:
    gr.Markdown("# 🛠️ Project 3: Smart Data Analytics Tool (COVID-19 Analysis)")
    file_input = gr.File(label="Upload COVID-19 CSV File here")
    submit_btn = gr.Button("Analyze Data", variant="primary")
    preview_out = gr.HTML(label="Dataset Preview")
    missing_out = gr.HTML(label="Missing Values")
    stats_out = gr.HTML(label="Statistical Summary")
    plot1 = gr.Plot(label="Chart 1")
    plot2 = gr.Plot(label="Chart 2")
    plot3 = gr.Plot(label="Chart 3")
    submit_btn.click(
        fn=analyze_csv,
        inputs=[file_input],
        outputs=[preview_out, missing_out, stats_out, plot1, plot2, plot3]
    )
tool.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9931d8d13963926ccc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
